In [2]:
import pandas as pd

df_processed = pd.read_json("../data/processed/corpus_experiment_A_internal.jsonl", lines=True)

In [3]:
df_processed.head()

,id,content,complexity,target_fields,ground_truth,schema_class,metadata,pdf_path,source
0,0ea66228-6efd-8386-8000-e2e0c72e3825.pdf,REMIT TO Sinclair Broadcast c/o WTTO PO Box 20...,L1,"{'advertiser': 'string', 'property': 'string',...",{'advertiser': 'Friends of Jeff Sessions Senat...,NaN,"{'word_count': 300, 'char_count': 2068, 'annot...",raw/vrdu/ad-buy-form/main/pdfs/0ea66228-6efd-8...,VRDU-adbuy
1,63ad699d-fa76-17bb-1bc5-ab2fafd0a2a3.pdf,Contract # Date Entered 04/15/20 Spots Schedul...,L1,"{'contract_num': 'string', 'flight_from': 'str...","{'contract_num': '4339717', 'flight_from': '05...",NaN,"{'word_count': 294, 'char_count': 1958, 'annot...",raw/vrdu/ad-buy-form/main/pdfs/63ad699d-fa76-1...,VRDU-adbuy
2,8ce03a7a-aa9c-da14-a16e-dabbec938595.pdf,Page 1 of 1 KMEG KMEG 100 Gold Circle Dr Dakot...,L1,"{'property': 'string', 'contract_num': 'string...","{'property': 'KMEG KMEG', 'contract_num': '135...",NaN,"{'word_count': 339, 'char_count': 2113, 'annot...",raw/vrdu/ad-buy-form/main/pdfs/8ce03a7a-aa9c-d...,VRDU-adbuy
3,a5b12494-83b6-c5c0-1e0f-0826f37899f0.pdf,Page 1 of 2 INVOICE FOX 31 Remit Address: Denv...,L1,"{'property': 'string', 'tv_address': 'string',...",{'property': 'Denver KDVR KDVR Denver KDVR KDV...,NaN,"{'word_count': 483, 'char_count': 2984, 'annot...",raw/vrdu/ad-buy-form/main/pdfs/a5b12494-83b6-c...,VRDU-adbuy
4,faa55a77-9090-22ac-fe9b-32ab3f026300.pdf,Contract # 26932832 CPE: //1447 Agency: AL MED...,L1,"{'contract_num': 'string', 'flight_to': 'strin...","{'contract_num': '26932832', 'flight_to': '6/8...",NaN,"{'word_count': 195, 'char_count': 1166, 'annot...",raw/vrdu/ad-buy-form/main/pdfs/faa55a77-9090-2...,VRDU-adbuy


In [6]:
from transformers import AutoTokenizer
import json
import pandas as pd

model_id = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Beispielprompt für die Tokenisierung
def get_full_prompt_tokens(row):
    prompt_text = f"Document: {row['content']}\nFields to extract: {json.dumps(row['target_fields'])}"
    return len(tokenizer.encode(prompt_text))

df_processed["input_token_count"] = df_processed.apply(get_full_prompt_tokens, axis=1)

p95 = df_processed["input_token_count"].quantile(0.95)
print(f"95% aller Dokumente benötigen weniger als {p95} Input-Tokens.")
print(f"Maximale Token Anzahl: {df_processed['input_token_count'].max()}")

95% aller Dokumente benötigen weniger als 6142.299999999998 Input-Tokens.
Maximale Token Anzahl: 9052


In [ ]:
def get_token_count(data):
    if not isinstance(data, dict):
        return 0
    return len(tokenizer.encode(json.dumps(data)))


df_processed["gt_tokens"] = df_processed["ground_truth"].apply(get_token_count)

stats = df_processed.groupby("complexity")["gt_tokens"].agg(
    ["mean", "median", "max", lambda x: x.quantile(0.95)]
)
stats.columns = ["Mean", "Median", "Max", "P95"]

print("Token-Bedarf der Ground Truth pro Komplexitätsstufe")
stats

--- Token-Bedarf der Ground Truth pro Komplexitätsstufe ---


,Mean,Median,Max,P95
complexity,,,,
L1,230.88,244.0,440,415.6
L2,664.92,636.0,1148,920.0
L3,1523.92,1447.0,3996,2394.2
